In [32]:
from core.settings import get_settings
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from services.calibrator_factory import CalibratorFactory
from services.model_registry import ModelRegistry
from pyi18next.backends.fs import Backend
from pyi18next.i18next import I18next
from utils.graph_builder import LocalizationGraphBuilder, traverse_namespaces
from adaptation.misc import NameAnonymizer
import os


In [33]:
settings = get_settings()
languages = settings.languages
print(settings)


languages={'es'} spacy={'es': 'es_core_news_sm'} sbert={'es': 'hiiamsid/sentence_similarity_spanish_es'} word2vec={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/vectors.bin'} siamese_lstm={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/bilstm_mean_cosine'} models=[<ModelType.SBERT: 'sbert'>, <ModelType.SPACY: 'spacy'>] allow_origins=['http://localhost:8080', 'http://127.0.0.1:8080'] host='0.0.0.0' port=8000 faiss_data_dir='./faiss_data' adaptation_data_dir='./adaptation/data' localization_dir='./adaptation/localization'


In [34]:
adaptation_dir = "./adaptation"

localization_dir = os.path.join(adaptation_dir, "localization")
language_dir = os.path.join(localization_dir, "dialogue", "active")
structure_dir = os.path.join(localization_dir,  "structure", "modified")

database_dir = "./faiss_data"

data_dir = os.path.join(adaptation_dir, "data")
name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")


In [35]:
namespaces = traverse_namespaces(language_dir, languages)
print(namespaces)


['scene6/routeA/scene6LunchRouteA', 'scene6/routeA/scene6BedroomRouteA2', 'scene6/routeB/scene6EndingRouteB', 'menus/titleScene', 'computer/socialMediaScreen', 'menus/creditsScene', 'scene1/scene1Bedroom2', 'dialogManager', 'computer/usernames', 'scene5/scene5Bedroom', 'scene6/routeA/scene6BedroomRouteA1', 'scene6/routeB/scene6BedroomRouteB', 'scene4/scene4Garage', 'deviceInfo', 'scene7/scene7Bedroom', 'scene2/scene2Break', 'scene6/scene6Bedroom', 'transitions', 'scene2/scene2Bedroom', 'generalDialogs', 'scene4/scene4Bedroom', 'scene1/scene1Lunch2', 'scene4/scene4Backyard', 'scene6/scene6Livingroom', 'scene6/routeA/scene6EndingRouteA', 'scene1/scene1Bedroom1', 'menus/loginScene', 'scene5/scene5Livingroom', 'scene6/routeB/scene6LunchRouteB', 'computer/captions', 'computer/loginScreen', 'scene1/scene1Lunch1', 'scene6/routeA/scene6PortalRouteA', 'scene1/scene1Break', 'scene3/scene3Break', 'scene4/scene4Frontyard', 'scene6/routeB/scene6PoliceStationRouteB', 'names', 'scene3/scene3Bedroom',

In [36]:
backend = Backend(
    name_mapping=lambda lng, ns: os.path.join(
        language_dir,
        lng,
        f"{ns}.json"
    )
)

i18n = I18next(
	backend=backend,
	lng=list(languages),
	ns=namespaces,
)



In [37]:
name_anonymizer = NameAnonymizer(
    names_path=spanish_names_path,
    whitelist_path=name_whitelist_path,
    replacement="[UNK]"
)


In [38]:
model_registry = ModelRegistry(languages)
model_registry.build_transformer("sbert")
model_registry.build_lstm()
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)
model_types = model_registry.active_model_types()


2026-07-05 05:33:47.597 | DEBUG    | services.model_registry:_create_loader:57 - Registering sbert loader for 'es'.
2026-07-05 05:33:47.600 | DEBUG    | services.model_registry:_create_loader:57 - Registering siamese_lstm loader for 'es'.
2026-07-05 05:33:47.601 | DEBUG    | services.model_registry:_create_loader:57 - Registering lstm calibrator loader for 'es'.
2026-07-05 05:33:47.602 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-07-05 05:33:49.106 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'
2026-07-05 05:33:49.106 | DEBUG    | services.lazy_loader:model:16 - Loading siamese_lstm for 'es'...


Using device: cuda


2026-07-05 05:33:50.526 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded siamese_lstm for 'es'
2026-07-05 05:33:50.527 | DEBUG    | services.lazy_loader:model:16 - Loading lstm calibrator for 'es'...
2026-07-05 05:33:50.536 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded lstm calibrator for 'es'


In [39]:
builder = LocalizationGraphBuilder(
    i18n=i18n,
    languages=languages,
    multilingual=multilingual,
    model_registry=model_registry,
    base_dir=structure_dir,
)

builder.run()


2026-07-05 05:33:52.446 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 33 vectors
2026-07-05 05:33:52.663 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 33 vectors
2026-07-05 05:33:54.990 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 40 vectors
2026-07-05 05:33:55.094 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 40 vectors
2026-07-05 05:33:56.982 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 31 vectors
2026-07-05 05:33:57.129 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 31 vectors
2026-07-05 05:33:59.060 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 48 vectors
2026-07-05 05:33:59.187 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 48 vectors
2026-07-05 05:33:59.202 | DEBUG    | services.node_engine:save_node:44 - Saving FAISS node | model=sbert | language=es | node=scene1Bedroom1_computer1_choices_similarity
2026-07-05 05:33:59.206 | DEBUG    | services.node_engine:save_node:44

Total visited nodes: 691


In [40]:
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)
test_engine = multilingual.get_node_engine("es", "sbert")

print(test_engine.retrievers)

test_engine.load_all()

print(test_engine.retrievers)


2026-07-05 05:33:59.233 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Bedroom1_computer1_choices_similarity
2026-07-05 05:33:59.235 | SUCCESS  | services.node_engine:load_node:71 - Loaded node successfully.
2026-07-05 05:33:59.235 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Bedroom1_computer2_root
2026-07-05 05:33:59.235 | SUCCESS  | services.node_engine:load_node:71 - Loaded node successfully.
2026-07-05 05:33:59.235 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Bedroom2_computer_choices2_similarity
2026-07-05 05:33:59.235 | SUCCESS  | services.node_engine:load_node:71 - Loaded node successfully.
2026-07-05 05:33:59.235 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks_similarity
2026-07-05 05:33:59.235 | SUCCESS  | 

{}
{'scene1Bedroom1_computer1_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001846EC518B0>, 'scene1Bedroom1_computer2_root': <controllers.retrievers.faiss.FaissRetriever object at 0x000001846EC539E0>, 'scene1Bedroom2_computer_choices2_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001846ECD92B0>, 'scene1Classroom_part2_thanks_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000001846ECD8B00>}


In [41]:
retriever = test_engine.get_retriever("scene1Classroom_part2_thanks_similarity")

retriever.search("Hola", 3)


(array([42, 27, 14], dtype=int32),
 array([0.99999994, 0.5333565 , 0.5254723 ], dtype=float32),
 array(['Hola', 'Buenas! Soy [UNK] encantado.', 'Holaaa, soy [UNK] que ta'],
       dtype=object))